In [ ]:
# Celda 1 - Librerias  [V0.6 - OpenCV Puro & GP Absoluto]
import cv2
import numpy as np
import time
import math
import threading
import collections
import customtkinter as ctk
from PIL import Image

# Librerias para Optimizador DEAP
import random
from deap import base, creator, tools, algorithms

ctk.set_appearance_mode('dark')
print('Celda 1 V0.6: Librerias base y DEAP listas.')

In [ ]:
# Celda 2 - Motor de vision con Contornos de Color  [V0.8]
from pygrabber.dshow_graph import FilterGraph
import cv2
import numpy as np
import time
import math

def detectar_camaras_sistema():
    try:
        graph = FilterGraph()
        return [(i, n) for i, n in enumerate(graph.get_input_devices())]
    except Exception as e:
        print(f'Error al buscar camaras: {e}')
        return []

RANGO_ROJO_1 = (np.array([  0, 100,  60]), np.array([ 12, 255, 255]))
RANGO_ROJO_2 = (np.array([168, 100,  60]), np.array([180, 255, 255]))
RANGO_NEGRO  = (np.array([  0,   0,   0]), np.array([180, 255,  60]))
RANGO_BLANCO = (np.array([  0,   0, 170]), np.array([180,  45, 255]))

LIMITES_PELOTAS = {'Rojo': 10, 'Negro': 10, 'Blanco': 10} # Hasta 10 de cualquier color
MAX_PELOTAS_TOTAL = 10

def get_color_bgr(nombre):
    n = nombre.lower()
    if 'rojo'   in n or 'red'   in n: return (0,   0, 220)
    if 'blanco' in n or 'white' in n: return (200, 200, 200)
    if 'negro'  in n or 'black' in n: return (80,  80,  80)
    return (0, 220, 220)

def estimar_z(radio_px, frame_shape):
    frac = (math.pi * radio_px * radio_px) / (frame_shape[0] * frame_shape[1])
    return round(max(0.1, min(5.0, 1.0 / (frac * 10 + 0.01))), 2)

class TrackerCirculo:
    def __init__(self, alpha=1.0, frames_conf=2, frames_perdida=5):
        self.alpha          = alpha
        self.frames_conf    = frames_conf
        self.frames_perdida = frames_perdida
        self.reiniciar()

    def reiniciar(self):
        self.suave       = None
        self.conteo_det  = 0
        self.conteo_perd = 0
        self.visible     = False

    def actualizar(self, deteccion):
        if deteccion is None:
            self.conteo_det  = 0
            self.conteo_perd = min(self.conteo_perd + 1, self.frames_perdida + 1)
            if self.conteo_perd >= self.frames_perdida:
                self.visible = False
                self.suave   = None
            return tuple(int(round(v)) for v in self.suave) if self.visible else None
        
        self.conteo_perd = 0
        self.conteo_det  = min(self.conteo_det + 1, self.frames_conf + 10)
        
        if self.conteo_det < self.frames_conf: 
            return None
            
        if self.suave is None:
            self.suave   = tuple(float(v) for v in deteccion[:3])
            self.visible = True
            return tuple(int(round(v)) for v in deteccion[:3])
            
        self.visible = True
        return tuple(int(round(v)) for v in deteccion[:3])

def emparejar_detecciones(trackers, detecciones):
    asignaciones = [None] * len(trackers)
    if not detecciones: return asignaciones
    det_usadas = set()
    for i, tr in enumerate(trackers):
        if tr.suave is not None and tr.visible:
            mejor_det = None
            mejor_dist = float('inf')
            for j, d in enumerate(detecciones):
                if j in det_usadas: continue
                dist = math.hypot(tr.suave[0] - d[0], tr.suave[1] - d[1])
                if dist < mejor_dist and dist < 120:
                    mejor_dist = dist
                    mejor_det = j
            if mejor_det is not None:
                asignaciones[i] = detecciones[mejor_det]
                det_usadas.add(mejor_det)
    for j, d in enumerate(detecciones):
        if j not in det_usadas:
            for i, tr in enumerate(trackers):
                if asignaciones[i] is None and (tr.suave is None or not tr.visible):
                    asignaciones[i] = d
                    break
    return asignaciones

def procesar_frame_vision(frame, temporizadores, trackers, callback_objeto=None):
    t_act       = time.time()
    hay_objetos = False
    fh, fw      = frame.shape[:2]
    detecciones_brutas = {'Rojo': [], 'Blanco': [], 'Negro': []}

    # --- FASE 1: Obtener Detecciones limitadas a la Malla Central ---
    cx_scr, cy_scr = fw // 2, fh // 2
    mesh_size = 180
    
    # Dibujar Malla visual de centralizacion en pantalla
    cv2.rectangle(frame, (cx_scr - mesh_size, cy_scr - mesh_size), (cx_scr + mesh_size, cy_scr + mesh_size), (255, 255, 255), 1)
    cv2.line(frame, (cx_scr, cy_scr - mesh_size), (cx_scr, cy_scr + mesh_size), (255, 255, 255), 1)
    cv2.line(frame, (cx_scr - mesh_size, cy_scr), (cx_scr + mesh_size, cy_scr), (255, 255, 255), 1)
    cv2.circle(frame, (cx_scr, cy_scr), mesh_size, (255, 255, 255), 1)

    # Mascara lógica: Todo lo que este fuera de esta malla negra será ignorado al 100%
    mask_centro = np.zeros((fh, fw), dtype=np.uint8)
    cv2.rectangle(mask_centro, (cx_scr - mesh_size, cy_scr - mesh_size), (cx_scr + mesh_size, cy_scr + mesh_size), 255, -1)

    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    
    # Las mascaras de color solo aplican dentro de la malla central (bitwise_and)
    mascaras = {
        'Rojo': cv2.bitwise_and(cv2.add(cv2.inRange(hsv, *RANGO_ROJO_1), cv2.inRange(hsv, *RANGO_ROJO_2)), mask_centro),
        'Negro': cv2.bitwise_and(cv2.inRange(hsv, *RANGO_NEGRO), mask_centro),
        'Blanco': cv2.bitwise_and(cv2.inRange(hsv, *RANGO_BLANCO), mask_centro)
    }

    # Recorremos cada mascara de color
    for nombre, mascara in mascaras.items():
        # Limpieza morfologica extrema para ignorar ruidos de fondo
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
        mascara = cv2.morphologyEx(mascara, cv2.MORPH_OPEN, kernel)
        mascara = cv2.morphologyEx(mascara, cv2.MORPH_CLOSE, kernel)
        
        contornos, _ = cv2.findContours(mascara, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        for contorno in contornos:
            area = cv2.contourArea(contorno)
            if area > 600:  # Evitar pixeles sueltos
                (cx, cy), radio = cv2.minEnclosingCircle(contorno)
                # Restringimos radio maximo a 160 para evitar detectar camisetas, paredes o sombras enormes
                if 20 < radio < 160:
                    perimetro = cv2.arcLength(contorno, True)
                    if perimetro > 0:
                        circularidad = 4 * math.pi * (area / (perimetro * perimetro))
                        # Tolerancia de circularidad baja un poco para admitir pelotas cortadas por el borde de la malla
                        if circularidad > 0.40: 
                            detecciones_brutas[nombre].append((int(cx), int(cy), int(radio)))

    # --- FASE 2: Emparejar y Dibujar ---
    total_pelotas_dibujadas = 0
    
    for nombre in ['Rojo', 'Blanco', 'Negro']:
        # Ordenamos pelotas de este color priorizando las mas cercanas al centro
        lista_detecciones = sorted(detecciones_brutas[nombre], key=lambda d: math.hypot(d[0]-cx_scr, d[1]-cy_scr))
        
        if total_pelotas_dibujadas >= MAX_PELOTAS_TOTAL:
            lista_detecciones = []
        else:
            cupo_restante = MAX_PELOTAS_TOTAL - total_pelotas_dibujadas
            lista_detecciones = lista_detecciones[:cupo_restante]
            
        asignaciones = emparejar_detecciones(trackers[nombre], lista_detecciones)
        
        for idx, det in enumerate(asignaciones):
            tr = trackers[nombre][idx]
            result = tr.actualizar(det)
            if result is None:
                temporizadores[nombre][idx] = 0.0
                continue
            
            if total_pelotas_dibujadas >= MAX_PELOTAS_TOTAL:
                break
                
            hay_objetos = True
            total_pelotas_dibujadas += 1
            cx, cy, r = result
            if temporizadores[nombre][idx] == 0.0: temporizadores[nombre][idx] = t_act
            
            color_bgr = get_color_bgr(nombre)
            x_norm = round(cx / fw * 2 - 1, 2)
            y_norm = round(1 - cy / fh * 2, 2)
            z_est  = estimar_z(r, frame.shape)
            
            # Dibujar el contorno preciso (sin lag)
            cv2.circle(frame, (cx, cy), r, color_bgr, 3) 
            cv2.circle(frame, (cx, cy), 4, color_bgr, -1)
                
            lbl = f'{nombre} #{idx+1}'
            lbl_c = f'X:{x_norm:+.1f} Y:{y_norm:+.1f} Z:{z_est}m'
            
            label_y = max(18, cy - r - 10)
            (tw, th), _ = cv2.getTextSize(lbl, cv2.FONT_HERSHEY_SIMPLEX, 0.55, 2)
            cv2.rectangle(frame, (cx - r, label_y - th - 4), (cx - r + tw + 4, label_y + 4), (20, 20, 20), -1)
            cv2.putText(frame, lbl, (cx - r + 2, label_y), cv2.FONT_HERSHEY_SIMPLEX, 0.55, color_bgr, 2)
            
            cy2 = label_y + th + 6
            (cw, ch), _ = cv2.getTextSize(lbl_c, cv2.FONT_HERSHEY_SIMPLEX, 0.42, 1)
            cv2.rectangle(frame, (cx - r, cy2 - ch - 2), (cx - r + cw + 4, cy2 + 2), (20, 20, 20), -1)
            cv2.putText(frame, lbl_c, (cx - r + 2, cy2), cv2.FONT_HERSHEY_SIMPLEX, 0.42, (210, 210, 210), 1)
            
            if callback_objeto is not None: callback_objeto(lbl, x_norm, y_norm, z_est)
            
    return frame, hay_objetos
print('Celda 2 V0.8 lista (Contornos en Malla).')

In [ ]:
# Celda 4 - Punto de entrada  [V0.6]
if __name__ == '__main__':
    app = EscanerApp()
    app.mainloop()